# Arkenstone Discovery V7 — transfer + mechanism campaign

Runs ARK-015 and ARK-016 under a 300-minute safety budget. ARK-015 tests non-arithmetic retention under presentation-distribution narrowing; ARK-016 separates literal LR from applied update magnitude and tests a trust-region candidate. The campaign emits a machine-readable training-design decision for the next scale.

Use a **T4 GPU** and run cells from top to bottom. Do not modify the pinned commit or thresholds after seeing results.


In [ ]:
import os, shutil, subprocess, sys, torch
PINNED_RUNNER_COMMIT = '4a8ffe84223fa64a7409b7d9936f2ff80d3828df'
REPO = '/content/An-Ra-the-new-AGI-discovery-v7'
RESULTS = '/content/arkenstone_discovery_v7_results'
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU before running.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
if os.path.exists(REPO): shutil.rmtree(REPO)
if os.path.exists(RESULTS): shutil.rmtree(RESULTS)
subprocess.run(['git','clone','--depth','50','--branch','Arkenstone','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_RUNNER_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_RUNNER_COMMIT, (head, PINNED_RUNNER_COMMIT)
paths = [
 'experiments/COLAB/discovery_v7_common.py',
 'experiments/COLAB/run_discovery_v7.py',
 'experiments/ARK-015/run_ark015.py',
 'experiments/ARK-016/run_ark016.py',
 'experiments/ARK-011/run_ark011.py',
 'experiments/ARK-014/run_ark014.py',
 'experiments/ARK-001/run_ark001.py',
 'experiments/lib/ark_tasks.py',
]
for p in paths:
    subprocess.run([sys.executable,'-m','py_compile',os.path.join(REPO,p)], check=True)
runner = os.path.join(REPO,'experiments/COLAB/run_discovery_v7.py')
env = dict(os.environ); env['PYTHONUNBUFFERED']='1'
subprocess.run([sys.executable, runner, '--smoke-test', '--expected-head', PINNED_RUNNER_COMMIT], cwd=REPO, env=env, check=True)
print('\nDISCOVERY V7 GPU SMOKE TEST PASS — safe to start the full campaign')


In [ ]:
# Full campaign. 300 minutes is a safety budget, not a minimum runtime.
import os, subprocess, sys
runner = os.path.join(REPO,'experiments/COLAB/run_discovery_v7.py')
env = dict(os.environ); env['PYTHONUNBUFFERED']='1'
proc = subprocess.run([sys.executable, runner, '--budget-minutes', '300', '--expected-head', PINNED_RUNNER_COMMIT], cwd=REPO, env=env)
print('FULL CAMPAIGN RETURN CODE:', proc.returncode)
if proc.returncode != 0:
    print('A failure receipt and all completed partial JSONs should be in the result ZIP. Do not rerun before inspecting them.')


In [ ]:
from pathlib import Path
root = Path('/content/arkenstone_discovery_v7_results')
print('Result files:')
for p in sorted(root.glob('*')):
    print(p.name, p.stat().st_size if p.is_file() else '')
decision = root / 'TRAINING_DESIGN_DECISION.json'
if decision.exists():
    print('\nTRAINING DESIGN DECISION:')
    print(decision.read_text())
zip_path = root / 'ARKENSTONE_DISCOVERY_V7_RESULTS.zip'
if zip_path.exists():
    print('\nZIP:', zip_path, zip_path.stat().st_size, 'bytes')
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print('Manual download: left Files panel → content → arkenstone_discovery_v7_results → ARKENSTONE_DISCOVERY_V7_RESULTS.zip', exc)
